# Unit 11 solutions: crashes, families of cars, and surviving them

These reference solutions measure what a crash costs, choose a fraction by the expected race time over the family, read racekit's perturbed family, build the respawn detector and the warning instruments, give Unit 10's agent its own fallback, and re-tune the fraction on a small training family.
Each exercise's statement is repeated, followed by a prediction made before running anything, then the solution.
Every race is seeded and uses `MPC_BUDGET_MS = None`, so every race number below is reproducible, and executable asserts check each answer, against an independent derivation where one exists.
Every number is a number about racekit's **carla-like car** and its perturbed family: a model, not the truth.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import PERTURBATION_FAMILY, carla_like, perturbation_factors
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_09, UNIT_10, UNIT_11 = AGENTS / "unit-09.py", AGENTS / "unit-10.py", AGENTS / "unit-11.py"
workdir = Path(mkdtemp(prefix="roar-unit-11-sol-"))
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
HOLD_S = 1.0  # s: the respawn hold (20 ticks)
NO_TIMING = {"MPC_BUDGET_MS": None}  # every MPC race here: no wall clock inside the race
TRAINING, HELD_OUT = tuple(range(24)), tuple(range(24, 48))  # the perturbed family's two halves
QUIET, CRASH = (0.0, 0.0, 0.0), (0.0, -1000.0, 0.0)  # N s: collision readings, nothing and racekit's crash impulse
MEASURED_MU_G_2026 = (7.70, 7.69, 7.70, 9.81, 8.16, 8.04)  # Unit 8's grip probes on test-circuit-seed2026 (m/s^2)


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, k=None, track="test-circuit", seed=None, lag_ticks=None, max_seconds=450.0):
    # One race on car k of the perturbed family (k = None: the unperturbed car), with its noise.
    # The race seed is k mod 5 unless one is given (0 for the unperturbed car).
    if seed is None:
        seed = 0 if k is None else k % 5
    car = carla_like(noise=True, perturb=k, lag_ticks=lag_ticks)
    return run_race(source, track, car=car, seed=seed, max_seconds=max_seconds)


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# Unit 9's line-relative clean rule; by default against test-circuit's line as the agents ship it.
with shim_installed():
    Unit11, _ = load_submission(UNIT_11)
line_offsets = np.array(Unit11(load_track("test-circuit").waypoints, None).offsets)


def line_error(result, offsets=None):
    offsets = line_offsets if offsets is None else offsets
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets=None):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR

## Exercise 1 - The cost of a crash

The lesson's rule: a crash at time t costs t + 1 s, to within a second, because the restart is a fresh race from rest.
Check it on a car the lesson did not race.
`agents/unit-11.py` at Unit 10's 0.58 (`variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)`) crashes once on held-out car 32, falls back, and finishes.
Race it on car 32 (`race(source, 32)`: race seed 2, `max_seconds=450.0`).

Its **fresh twin** is the same car and race seed, started in the fallback configuration: `variant(UNIT_11, "u11-twin", STEERING="pursuit", MU_FRACTION=0.58 * 0.86, **NO_TIMING)`.
Race the twin with `repeat_races` on `"test-circuit"`, `n=3, seeds=(2, 3, 4)`, `car_factory=lambda: carla_like(noise=True, perturb=32)` and `max_seconds=450.0`: its first race is the same-seed twin, and the three together show the noise.

Write:

- `crash_costs(result)`: the list of t + `HOLD_S` for every respawn time t of a race result (`result.respawn_times`).

Define:

- `crash_costs`: the function;
- `crash_race`: the race result, and `crash_time`: its first respawn time (s);
- `held_rows`: the number of telemetry rows with `held == 1`, and `hold_seconds`: `held_rows * DT`;
- `restart_progress`: the `progress` of the first telemetry row after the crash with `held == 0`;
- `remainder`: the race time after the restart, `crash_race.total_time - crash_time - hold_seconds`;
- `twin_results`: the `RepeatResult`, `twin_time`: its first race's time, and `twin_spread`: its spread (s);
- `twin_gap`: `remainder - twin_time`, and `within_a_second`: `True` when |`twin_gap`| ≤ 1.0;
- `systematic`: `True` when |`twin_gap`| is larger than `twin_spread` (a difference the noise does not explain);
- `fig`, `ax`: `progress` against `t` for the crashed race and the same-seed twin, with a legend, axis labels and a title;
- `why_shorter`: two sentences: the remainder is a little shorter than the fresh twin; what does racekit's runner reset at a respawn, what does it not, and which part of the agent carries over?

**Prediction.** The hold is 20 rows (1 s), the first row after it is at progress 0, and the race after the restart should take within a second of the fresh twin, a little less (the lesson's 0.40 to 0.65 s), because the speed loop's integral carries over.
Pure pursuit's noise spread over three seeds is a few tenths of a second, so a gap of half a second may well be larger than it.

In [ ]:
def crash_costs(result):
    return [t + HOLD_S for t in result.respawn_times]


crash_source = variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)
crash_race = race(crash_source, 32)
crash_time = crash_race.respawn_times[0]
telemetry = crash_race.telemetry
held_rows = int(np.sum(telemetry["held"] == 1))
hold_seconds = held_rows * DT
after = telemetry[(telemetry["t"] > crash_time) & (telemetry["held"] == 0)]
restart_progress = int(after["progress"][0])
remainder = crash_race.total_time - crash_time - hold_seconds

twin_source = variant(UNIT_11, "u11-twin", STEERING="pursuit", MU_FRACTION=0.58 * 0.86, **NO_TIMING)
twin_results = repeat_races(twin_source, "test-circuit", n=3, seeds=(2, 3, 4),
                            car_factory=lambda: carla_like(noise=True, perturb=32), max_seconds=450.0)
twin_time = float(twin_results.times[0])
twin_spread = twin_results.spread
twin_gap = remainder - twin_time
within_a_second = abs(twin_gap) <= 1.0
systematic = abs(twin_gap) > twin_spread

twin = twin_results.results[0]
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(telemetry["t"], telemetry["progress"], label="car 32, crashed (Unit 11 at 0.58)")
ax.plot(twin.telemetry["t"], twin.telemetry["progress"], label="fresh twin (pursuit, 0.58 x 0.86), seed 2")
ax.plot(twin.telemetry["t"] + crash_time + hold_seconds, twin.telemetry["progress"], linestyle="--",
        label="the same twin, started at the restart")
ax.set_xlabel("t (s)")
ax.set_ylabel("progress (waypoints)")
ax.set_title("Held-out car 32: the race after the restart is a fresh race, to within a second")
ax.legend()
plt.show()
why_shorter = (
    "At a respawn racekit's runner resets the car (back to the start pose, at rest) and the race's progress, and "
    "holds it for 1 s, but it does not reset the agent or re-seed the noise, so the restart is a new draw of the noise "
    "driven by the same agent object. "
    "Unit 11's fall_back() clears the MPC's memory, but the speed loop's PI integral built up before the crash carries "
    f"over, so the restarted car takes a little more throttle at first than a fresh one and finishes {-twin_gap:.2f} s "
    "sooner than its same-seed twin."
)

print(f"crash at {crash_time:.2f} s ({crash_race.respawn_causes[0]}); collisions {crash_race.collisions}, "
      f"finished {crash_race.finished} in {crash_race.total_time:.2f} s; crash costs {crash_costs(crash_race)}")
print(f"held rows {held_rows} ({hold_seconds:.2f} s); restart progress {restart_progress}; remainder {remainder:.2f} s")
print(f"twin times {twin_results.times}, spread {twin_spread:.2f} s; gap {twin_gap:+.2f} s; "
      f"within a second {within_a_second}; systematic {systematic}")
print(why_shorter)

# independent checks: the hold is 20 ticks of 0.05 s; the restart starts from progress 0; the race time splits as
# crash + hold + remainder; the twin is a fresh race of the same car and seed in the fallback configuration
assert crash_race.finished and crash_race.collisions == 1 and abs(crash_time - 103.05) <= TICK
assert crash_costs(crash_race) == [crash_time + HOLD_S]
assert held_rows == 20 and abs(hold_seconds - HOLD_S) <= 1e-12 and restart_progress == 0
assert abs(crash_race.total_time - 389.85) <= TICK and abs(remainder - 285.80) <= TICK
assert abs(twin_time - 286.30) <= TICK and abs(twin_spread - 0.25) <= TICK
assert within_a_second and twin_gap < 0 and systematic  # shorter than the twin, by more than the noise

## Exercise 2 - Expected race time, and the best fraction

The book's tests (plan `roar-010`, premise U11.2 and U11.4) raced Unit 10's agent at four fractions on the 24 training cars of the family (race seed k mod 5, a 700 s limit).
The cell below gives, per fraction, how many races were clean and their mean time, and every race that was not clean (car, race time, finished).
A race counts at **min(its time, the cap)** if it finished and at the cap if it did not: with a lower cap, a slow finish becomes a DNF.

Write:

- `capped_time(time, finished, cap)`: one race's time as counted;
- `expected_time(n_clean, clean_mean, not_clean, cap)`: the mean over all the races of a fraction, the clean ones at `clean_mean` (their mean is all you need) and each not-clean one by `capped_time`.

Define:

- `capped_time`, `expected_time`: the functions;
- `expected_700`: a dict from each fraction to its expected race time with a 700 s cap, and `matches_premise`: `True` when every one is within 0.01 s of `PREMISE_MEAN`;
- `best_700`: the fraction with the shortest expected race time;
- `expected_450`, `best_450`: the same with a 450 s cap (assume every clean race is under 450 s);
- `step_gain`: how much faster a clean race is at 0.56 than at 0.54 (from `CLEAN`), and `crash_cost_056`: car 8's race time at 0.56 minus the clean mean at 0.56;
- `break_even`: `step_gain / crash_cost_056`, the crash rate per race below which 0.56 would beat 0.54, and `races_per_crash`: `1 / break_even`;
- `fig`, `ax`: the expected race time against the fraction for both caps, with a legend, axis labels and a title;
- `why_not_the_edge`: two sentences: why does a fraction one step below the highest clean one win on expected time, and why can 24 cars not tell you whether the faster fraction pays?

**Prediction.** With the 700 s cap the expected times are the premise's: 318.66, 294.35, 279.95 and 282.58 s, best at 0.54.
A 450 s cap makes every not-clean race count at most 450 s, which helps 0.58 and 0.56 a lot, but they still carry 2 or 3 races near 450 s against clean races only about 2.5 s faster, so 0.54 should still win.
The gain of one step is 279.95 − 277.43 = 2.52 s and car 8's crash at 0.56 cost 474.00 − 277.43 = 196.57 s: about one race in 78.

In [ ]:
CAP = 700.0  # s: the book's tests' cap
FAMILY_SIZE = 24
# plan roar-010, premise U11.2/U11.4: Unit 10 on training cars 0-23 (race seed k mod 5, 700 s cap).
CLEAN = {0.58: (21, 275.05), 0.56: (22, 277.43), 0.54: (24, 279.95), 0.52: (24, 282.58)}  # (count, mean time in s)
NOT_CLEAN = {  # (car, race time in s, finished)
    0.58: [(8, 471.85, True), (16, 700.0, False), (20, 700.0, False)],
    0.56: [(8, 474.00, True), (9, 486.90, True)],
    0.54: [],
    0.52: [],
}
PREMISE_MEAN = {0.58: 318.66, 0.56: 294.35, 0.54: 279.95, 0.52: 282.58}  # s: the premise's means (700 s cap)



def capped_time(time, finished, cap):
    return min(time, cap) if finished else cap


def expected_time(n_clean, clean_mean, not_clean, cap):
    total = n_clean * clean_mean + sum(capped_time(time, finished, cap) for _, time, finished in not_clean)
    return total / (n_clean + len(not_clean))


expected_700 = {f: expected_time(*CLEAN[f], NOT_CLEAN[f], CAP) for f in CLEAN}
matches_premise = all(abs(expected_700[f] - PREMISE_MEAN[f]) <= 0.01 for f in CLEAN)
best_700 = min(expected_700, key=expected_700.get)
expected_450 = {f: expected_time(*CLEAN[f], NOT_CLEAN[f], 450.0) for f in CLEAN}
best_450 = min(expected_450, key=expected_450.get)
step_gain = CLEAN[0.54][1] - CLEAN[0.56][1]
crash_cost_056 = [time for k, time, _ in NOT_CLEAN[0.56] if k == 8][0] - CLEAN[0.56][1]
break_even = step_gain / crash_cost_056
races_per_crash = 1 / break_even

fractions = sorted(CLEAN)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(fractions, [expected_700[f] for f in fractions], "o-", label="cap 700 s")
ax.plot(fractions, [expected_450[f] for f in fractions], "s--", label="cap 450 s")
ax.set_xlabel("MU_FRACTION")
ax.set_ylabel("expected race time (s)")
ax.set_title("Unit 10 on the 24 training cars: expected race time against the fraction")
ax.legend()
plt.show()
why_not_the_edge = (
    f"One step of fraction makes a clean race only {step_gain:.2f} s faster, while one crash costs about "
    f"{crash_cost_056:.0f} s, so the higher fraction pays only if fewer than {break_even:.1%} of races crash (one in "
    f"{races_per_crash:.0f}), and at 0.56 two training cars of 24 already crash: one step below the highest clean "
    "fraction gives up a little on every race to avoid the crashes that cost almost a race each. "
    "With 24 cars the smallest crash rate you can see is one in 24, about 4%, three times the break-even rate, so "
    "24 cars clean at a fraction cannot show that its crash rate is below 1.3%, and the faster fraction's pay-off "
    "stays unmeasured."
)

for f in fractions:
    print(f"f = {f}: expected {expected_700[f]:.2f} s (700 s cap), {expected_450[f]:.2f} s (450 s cap)")
print(f"best: {best_700} (700), {best_450} (450); step gain {step_gain:.2f} s, crash cost {crash_cost_056:.2f} s; "
      f"break-even {break_even:.4f}, one race in {races_per_crash:.1f}")
print(why_not_the_edge)

# independent checks: the cell's own data, summed by hand
assert capped_time(471.85, True, 450.0) == 450.0 and capped_time(300.0, True, 450.0) == 300.0
assert capped_time(700.0, False, 700.0) == 700.0
assert abs(expected_700[0.58] - (21 * 275.05 + 471.85 + 700.0 + 700.0) / 24) <= 1e-9
assert matches_premise and best_700 == 0.54 and best_450 == 0.54
assert abs(expected_450[0.58] - (21 * 275.05 + 3 * 450.0) / 24) <= 1e-9
assert abs(expected_450[0.56] - (22 * 277.43 + 2 * 450.0) / 24) <= 1e-9
assert expected_450[0.54] == expected_700[0.54] and expected_450[0.52] == expected_700[0.52]
assert abs(step_gain - 2.52) <= 1e-9 and abs(crash_cost_056 - 196.57) <= 1e-9
assert abs(break_even - 2.52 / 196.57) <= 1e-12 and round(races_per_crash) == 78

## Exercise 3 - The perturbed family

`perturbation_factors(k)` returns car k's six factors (a dict from the names in `PERTURBATION_FAMILY` to a number, 1.0 meaning the car Unit 8 measured); `PERTURBATION_FAMILY` maps each name to its (low, high) range.

Define:

- `factors`: a dict from k = 0 … 47 to `perturbation_factors(k)`;
- `inside_ranges`: `True` when every factor of every car lies inside its range;
- `same_again`: `True` when `perturbation_factors(5)` equals `factors[5]` (drawn again);
- `grip_order`: the training cars sorted by their grip factor, lowest first;
- `effective_fraction`: a dict from each training car to `0.58 / grip`: the share of **that car's** grip that a planner using 0.58 of the **measured** grip really asks for (Unit 8's measured grip is the unperturbed car's);
- `over_edge`: the training cars whose effective fraction is above 0.60 (MPC's family edge in Unit 10), in `grip_order`.

Then race Unit 10's agent as shipped (0.58, `variant(UNIT_10, "u10-0.58", **NO_TIMING)`) on the **two** lowest-grip training cars, `grip_order[:2]`, each with `race(source, k, max_seconds=300.0)` (a clean race takes about 277 s).

- `family_races`: a dict from those two cars to their race results, and `crashed`: a dict from each to `result.collisions > 0`;
- `crash_ranks`: a dict from each car of `PREMISE_CRASHED_058` (the cars that were not clean at 0.58 in the book's tests) to its position in `grip_order` (1 = the lowest grip);
- `fig`, `ax`: a scatter of grip factor (x) against throttle factor (y) for the training cars, with the cars of `PREMISE_CRASHED_058` drawn with a different marker, a legend, axis labels and a title;
- `grip_story`: two sentences: does the grip factor alone predict which cars crash at 0.58? Use your two races and the throttle factors.

**Prediction.** The factors come from a generator seeded by k, so they are inside their ranges and the same when drawn again.
The two lowest-grip training cars are 16 and 18 (the lesson's list); on car 16 the planner uses 0.58 / 0.930 ≈ 0.62 of the grip, over 0.60, and Unit 10 crashes it at about 190.6 s, while car 18, almost as low in grip, should be clean.

In [ ]:
PREMISE_CRASHED_058 = (8, 16, 20)  # not clean at 0.58 in the book's tests (premise U11.2)
factors = {k: perturbation_factors(k) for k in range(48)}
inside_ranges = all(low <= factors[k][name] <= high
                    for k in factors for name, (low, high) in PERTURBATION_FAMILY.items())
same_again = perturbation_factors(5) == factors[5]
grip_order = sorted(TRAINING, key=lambda k: factors[k]["grip"])
effective_fraction = {k: 0.58 / factors[k]["grip"] for k in TRAINING}
over_edge = [k for k in grip_order if effective_fraction[k] > 0.60]

unit10_058 = variant(UNIT_10, "u10-0.58", **NO_TIMING)
family_races = {k: race(unit10_058, k, max_seconds=300.0) for k in grip_order[:2]}
crashed = {k: result.collisions > 0 for k, result in family_races.items()}
crash_ranks = {car: rank for rank, car in enumerate(grip_order, start=1) if car in PREMISE_CRASHED_058}

others = [k for k in TRAINING if k not in PREMISE_CRASHED_058]
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter([factors[k]["grip"] for k in others], [factors[k]["throttle"] for k in others], label="clean at 0.58")
ax.scatter([factors[k]["grip"] for k in PREMISE_CRASHED_058], [factors[k]["throttle"] for k in PREMISE_CRASHED_058],
           marker="x", s=80, label="not clean at 0.58 (the book's tests)")
ax.axvline(0.58 / 0.60, color="grey", linestyle="--", label="0.58 / grip = 0.60")
ax.set_xlabel("grip factor")
ax.set_ylabel("throttle factor")
ax.set_title("The 24 training cars: grip against throttle")
ax.legend()
plt.show()
grip_story = (
    f"Not alone: car 16, the lowest grip ({factors[16]['grip']:.3f}), crashed at 0.58 (at "
    f"{family_races[16].respawn_times[0]:.2f} s, then again), but car 18, almost as low "
    f"({factors[18]['grip']:.3f}, also over the 0.60 edge at {effective_fraction[18]:.3f}), was clean in "
    f"{family_races[18].total_time:.2f} s, and {len(over_edge)} training cars are over the edge while only 3 crash, "
    f"car 20 ranking only 5th by grip. "
    f"Car 18's throttle factor is {factors[18]['throttle']:.3f}, about 9% weak, so it reaches every corner a little "
    f"slower, while car 16's is {factors[16]['throttle']:.3f} and car 20's {factors[20]['throttle']:.3f}: the "
    "factors act together, so a crash is decided by the whole car, and the only test is to race it."
)

print("grip order:", grip_order)
print("over the edge:", over_edge, {k: round(effective_fraction[k], 3) for k in over_edge})
for k in grip_order[:6]:
    print(k, {name: round(value, 3) for name, value in factors[k].items()})
for k, result in family_races.items():
    print(f"car {k}: collisions {result.collisions} at {result.respawn_times}, finished {result.finished}, "
          f"{result.total_time:.2f} s, clean {clean(result)}")
print("crash ranks:", crash_ranks)
print(grip_story)

# independent checks: the ranges, the effective fraction by hand, and the order by grip
assert len(factors) == 48 and inside_ranges and same_again
assert grip_order[:6] == [16, 18, 8, 9, 20, 13] and len(grip_order) == 24
assert all(factors[a]["grip"] <= factors[b]["grip"] for a, b in zip(grip_order, grip_order[1:]))
assert abs(effective_fraction[16] - 0.58 / factors[16]["grip"]) <= 1e-12 and effective_fraction[16] > 0.62
assert over_edge == [k for k in grip_order if factors[k]["grip"] < 0.58 / 0.60] and len(over_edge) == 13
assert crashed == {16: True, 18: False}
assert [round(t, 2) for t in family_races[16].respawn_times] == [190.60, 291.90]  # the lesson's hook car 16
assert clean(family_races[18]) and abs(family_races[18].total_time - 276.60) <= TICK
assert crash_ranks == {8: 3, 16: 1, 20: 5}
assert factors[18]["throttle"] < 0.91 and factors[16]["throttle"] > 1.0

## Exercise 4 - A respawn detector

Nobody tells the agent that it crashed; it has its collision sensor, `self.collision_sensor.get_last_gym_observation()`, a 3-vector impulse (N·s).
racekit reports the crash impulse on the first step after the hold, on that one step; CARLA's sensor may repeat its last reading.

Write the lesson's edge rule twice.

First as a function, `edge_rule(readings, threshold=100.0, refractory=40)`: given one reading per step (steps counted from 1), return the list of steps on which a respawn is handled: the reading's norm (`np.linalg.norm`) is over `threshold`, the reading is not equal to the previous step's, and no respawn was handled in the last `refractory` steps (a step s is allowed when s − the last handled step ≥ `refractory`).

Then inside an agent.
Load Unit 10 at 0.58 (`with shim_installed(): Unit10At058, _ = load_submission(variant(UNIT_10, "u10-0.58", **NO_TIMING))`) and write `class Detector(Unit10At058)` whose `__init__` calls `super().__init__(*args, **kwargs)`, sets up its own memory (a step counter, the previous reading, the step of the last respawn handled, and `self.handled`, a list of the steps it handled) and appends itself to a list `detectors`; and whose `async def step(self)` counts the step, applies the rule to this step's reading (as a tuple of floats), records a handled step, and then does everything Unit 10 does: `await super().step()`.
Race `Detector` on car 20 (`race(Detector, 20, max_seconds=300.0)`): Unit 10 crashes twice there in 300 s.

Define:

- `edge_rule`, `Detector`, `detectors`: the function, the class and the list;
- `script_handled`: `edge_rule(SCRIPT)`, and `script_level`: the number of readings in `SCRIPT` whose norm is over 100 (what a level-triggered detector would count);
- `detector_race`: the race result, and `detector`: the agent (`detectors[0]`);
- `count_matches`: `True` when `len(detector.handled)` equals `detector_race.collisions`;
- `detection_times`: the time of each handled step (the step-th driving row, `held == 0`, of the telemetry: one row per `step()` call), and `delays`: each detection time minus its respawn time (s);
- `why_edge`: one sentence: what would a level-triggered detector do on CARLA, and what does the 40-step window add to "not the previous reading"?

**Prediction.** On `SCRIPT` the edge rule handles step 4 (the first crash reading), not steps 5 to 11 (repeats), not step 22 (18 steps after 4), and step 63 (59 steps after 4, and a change from the quiet step before it): `[4, 63]`, where a level-triggered detector counts 8 + 1 + 2 = 11.
On car 20 the detector should handle each of the two respawns once, on the first step after the 1 s hold, so each detection comes about 1 s after its respawn time.

In [ ]:
SCRIPT = [QUIET] * 3 + [CRASH] * 8 + [QUIET] * 10 + [CRASH] + [QUIET] * 40 + [CRASH] * 2  # one reading per step


def edge_rule(readings, threshold=100.0, refractory=40):
    handled, previous, last = [], None, None
    for step, reading in enumerate(readings, start=1):
        edge = float(np.linalg.norm(reading)) > threshold and reading != previous
        if edge and (last is None or step - last >= refractory):
            handled.append(step)
            last = step
        previous = reading
    return handled


detectors = []
with shim_installed():
    Unit10At058, _ = load_submission(variant(UNIT_10, "u10-0.58", **NO_TIMING))


class Detector(Unit10At058):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.steps = 0  # step() calls so far
        self.previous_reading = None
        self.last_handled = None  # the step of the last respawn handled
        self.handled = []
        detectors.append(self)

    async def step(self):
        self.steps += 1
        reading = tuple(float(value) for value in self.collision_sensor.get_last_gym_observation())
        edge = float(np.linalg.norm(reading)) > 100.0 and reading != self.previous_reading
        if edge and (self.last_handled is None or self.steps - self.last_handled >= 40):
            self.handled.append(self.steps)
            self.last_handled = self.steps
        self.previous_reading = reading
        await super().step()


script_handled = edge_rule(SCRIPT)
script_level = sum(1 for reading in SCRIPT if float(np.linalg.norm(reading)) > 100.0)
detector_race = race(Detector, 20, max_seconds=300.0)
detector = detectors[0]
count_matches = len(detector.handled) == detector_race.collisions
driving_t = detector_race.telemetry["t"][detector_race.telemetry["held"] == 0]
detection_times = [float(driving_t[step - 1]) for step in detector.handled]
delays = [detected - respawn for detected, respawn in zip(detection_times, detector_race.respawn_times)]
why_edge = (
    f"A level-triggered detector on CARLA would count every step on which the sensor repeats its last reading as a new "
    f"respawn ({script_level} on SCRIPT, for 2 real events), falling back again and again; \"not the previous "
    "reading\" removes a repeated reading, and the 40-step window also removes a reading that flickers back to the "
    "crash value after a quiet step (step 22 of SCRIPT), while two real respawns, which cannot come 2 s apart, are "
    "still both handled."
)

print(f"script: handled {script_handled}, level-triggered count {script_level}")
print(f"car 20: collisions {detector_race.collisions} at {detector_race.respawn_times}; handled steps {detector.handled}; "
      f"detected at {detection_times}; delays {delays}; steps {detector.steps}, driving rows {len(driving_t)}")
print(why_edge)

# independent checks: SCRIPT by hand; on car 20 one handled step per respawn, one hold (20 ticks) plus a tick after it
assert script_handled == [4, 63] and script_level == 8 + 1 + 2
assert edge_rule([QUIET, CRASH, QUIET, CRASH]) == [2] and edge_rule([CRASH] * 3) == [1]
assert [round(t, 2) for t in detector_race.respawn_times] == [99.10, 294.65] and count_matches
assert detector.steps == len(driving_t)  # one driving row per step() call
assert all(abs(delay - (HOLD_S + DT)) <= 1e-6 for delay in delays)

## Exercise 5 - Sideslip and line error, logged

The sideslip β is the angle from the car's heading to its velocity: β = atan2(v_y, v_x) − ψ, wrapped to [−180°, 180°), with (v_x, v_y) the velocity sensor's reading and ψ the yaw (`self.rpy_sensor.get_last_gym_observation()[2]`).
`agents/unit-11.py` computes it every step as `self.beta` (degrees), and its line error is `self.cross_track_error` (m).

Write:

- `my_beta(velocity, yaw)`: β in degrees from a velocity reading and a yaw (rad), wrapped with `(angle + math.pi) % (2 * math.pi) - math.pi` before converting;
- `logger(source)`: returns `(Logged, log)`, where `Logged` is a subclass of the submission's class (loaded with `load_submission` inside `shim_installed()`) whose `step` first awaits `super().step()` and then appends to `log`, a dict of lists: `"beta"` (`self.beta`), `"mine"` (`my_beta` of this step's velocity and yaw readings), `"error"` (`self.cross_track_error`) and `"speed"` (the velocity's norm).

With `source = variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)`, race a fresh `Logged` on car 16 (`max_seconds=195.0`: it crashes at about 190.6 s) and another on car 0 (`max_seconds=450.0`: a clean race).
Turn each log into arrays, and add `"t"`: the telemetry's `t` on the driving rows (`held == 0`), one per step.

Define:

- `my_beta`, `logger`: the functions;
- `crash_race`, `crash_log`, `clean_race`, `clean_log`: the two races and their logs (dicts of arrays);
- `beta_agrees`: `True` when `my_beta` and `self.beta` differ by less than 1e-9 deg on every step of both races;
- `peak_beta_clean`, `peak_error_clean`: the largest |β| (deg) and |line error| (m) of the clean race, over the steps above 5 m/s;
- `lead_beta`, `lead_error`: the **lead time** of |β| over 8° and of |line error| over 1.6 m before car 16's crash (its first respawn time): how long before the crash the **last** run of consecutive steps (above 5 m/s) over the threshold began, the run that a warning would be part of (s; `None` if never over it);
- `fig`, `(ax_beta, ax_error)`: β and the line error against the time to the crash over the last 10 s before it, each with its warning threshold (±) and the clean race's peak (±) as horizontal lines, axis labels and titles;
- `enough_time`: two sentences: at 30 m/s, how far does the car travel in `lead_beta`, and what could an agent still do in that distance?

**Prediction.** `my_beta` is the agent's formula, so it should agree to rounding, as long as the readings after `super().step()` are the ones the step used (the race advances the car only after the step returns).
These are the lesson's car-16 race and a new clean car; I expect the β warning about 1.1 s before car 16's crash and the line-error warning only a few tenths of a second before it, with the clean race's peaks near the lesson's car 1 (about 5° and 1.3 m).

In [ ]:
def my_beta(velocity, yaw):
    angle = math.atan2(float(velocity[1]), float(velocity[0])) - yaw
    return math.degrees((angle + math.pi) % (2 * math.pi) - math.pi)


def logger(source):
    with shim_installed():
        Agent, _ = load_submission(source)
    log = {"beta": [], "mine": [], "error": [], "speed": []}

    class Logged(Agent):
        async def step(self):
            await super().step()
            velocity = self.velocity_sensor.get_last_gym_observation()
            yaw = float(self.rpy_sensor.get_last_gym_observation()[2])
            log["beta"].append(self.beta)
            log["mine"].append(my_beta(velocity, yaw))
            log["error"].append(self.cross_track_error)
            log["speed"].append(float(np.linalg.norm(velocity)))

    return Logged, log


def as_arrays(result, log):
    arrays = {name: np.array(values) for name, values in log.items()}
    arrays["t"] = result.telemetry["t"][result.telemetry["held"] == 0]
    return arrays


def lead_time(log, name, threshold, t_crash):
    # how long before the crash the last run of steps over the threshold (above 5 m/s) began; None if never
    over = (np.abs(log[name]) > threshold) & (log["speed"] > 5.0) & (log["t"] < t_crash)
    if np.sum(over) == 0:
        return None
    start = int(np.nonzero(over)[0][-1])
    while start > 0 and over[start - 1]:
        start -= 1
    return float(t_crash - log["t"][start])


source = variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)
Logged, log = logger(source)
crash_race = race(Logged, 16, max_seconds=195.0)
crash_log = as_arrays(crash_race, log)
Logged, log = logger(source)
clean_race = race(Logged, 0, max_seconds=450.0)
clean_log = as_arrays(clean_race, log)

beta_agrees = all(np.max(np.abs(run["mine"] - run["beta"])) < 1e-9 for run in (crash_log, clean_log))
moving = clean_log["speed"] > 5.0
peak_beta_clean = float(np.max(np.abs(clean_log["beta"][moving])))
peak_error_clean = float(np.max(np.abs(clean_log["error"][moving])))
t_crash = crash_race.respawn_times[0]
lead_beta = lead_time(crash_log, "beta", 8.0, t_crash)
lead_error = lead_time(crash_log, "error", 1.6, t_crash)

window = (crash_log["t"] > t_crash - 10.0) & (crash_log["t"] < t_crash)
fig, (ax_beta, ax_error) = plt.subplots(1, 2, figsize=(11, 3.5))
ax_beta.plot(crash_log["t"][window] - t_crash, crash_log["beta"][window], label="car 16")
ax_error.plot(crash_log["t"][window] - t_crash, crash_log["error"][window], label="car 16")
for ax, limit, peak, name in ((ax_beta, 8.0, peak_beta_clean, "sideslip beta (deg)"),
                              (ax_error, 1.6, peak_error_clean, "line error (m, left +)")):
    for sign in (1, -1):
        ax.axhline(sign * limit, color="grey", linestyle="--", label="warning" if sign == 1 else None)
        ax.axhline(sign * peak, color="green", linewidth=0.8, label="clean race's peak (car 0)" if sign == 1 else None)
    ax.set_xlabel("time to the crash (s)")
    ax.set_ylabel(name)
    ax.legend()
ax_beta.set_title("Car 16's sideslip, the last 10 s before its crash")
ax_error.set_title("Car 16's line error, the last 10 s before its crash")
fig.tight_layout()
plt.show()
enough_time = (
    f"At 30 m/s the car covers 30 x {lead_beta:.2f} = {30 * lead_beta:.0f} m between the 8 deg sideslip warning and "
    "the crash, about the length of a corner's entry. "
    f"In that distance the agent could still brake (at Unit 7's {5.0:.0f} m/s^2 it would shed only about "
    f"{5.0 * lead_beta:.1f} m/s, from 30 to about {30 - 5.0 * lead_beta:.1f}) or ease its steering, but a car already "
    "sliding at more than 8 deg may not answer in time, so it can soften the crash's odds, not much more."
)

print(f"car 16: crash at {t_crash:.2f} s; steps {len(crash_log['beta'])}, driving rows {len(crash_log['t'])}")
print(f"car 0: {clean_race.total_time:.2f} s, clean {clean(clean_race)}; steps {len(clean_log['beta'])}, rows {len(clean_log['t'])}")
print(f"beta agrees {beta_agrees}; clean peaks {peak_beta_clean:.2f} deg, {peak_error_clean:.2f} m; leads {lead_beta}, {lead_error}")
print(enough_time)

# independent checks: one log entry per step() call, the two beta formulas, and the lesson's car-16 lead times
for log_arrays in (crash_log, clean_log):
    assert len(log_arrays["beta"]) == len(log_arrays["t"]) == len(log_arrays["mine"])
assert beta_agrees and abs(my_beta((1.0, 1.0), 0.0) - 45.0) <= 1e-12 and abs(my_beta((0.0, -1.0), math.pi / 2) + 180.0) <= 1e-9
assert abs(t_crash - 190.60) <= TICK and crash_race.collisions == 1
assert clean(clean_race) and abs(clean_race.total_time - 274.55) <= TICK
assert abs(peak_beta_clean - 5.00) <= 0.01 and abs(peak_error_clean - 1.23) <= 0.01
assert peak_beta_clean < 8.0 and peak_error_clean < 1.6  # no warning at all in the clean race
assert abs(lead_beta - 1.10) <= TICK and abs(lead_error - 0.25) <= TICK

## Exercise 6 - A warning threshold table

A warning is useful when it comes **early** before a crash and **never** when nothing is wrong.
Use Exercise 5's two logs (no new races).

For a threshold, the **lead time** is how long before car 16's crash the last run over it began (as in Exercise 5: a crossing early in the race, followed by a return below the threshold, is not a warning of this crash), and the **false alarms** are the steps of the clean race (above 5 m/s) on which it was over it.
A run of consecutive false-alarm steps is one **episode**.

Define:

- `beta_table`: a dict from each β threshold in `BETA_THRESHOLDS` to `(lead time or None, false-alarm steps, false-alarm episodes)`;
- `error_table`: the same for the line error and `ERROR_THRESHOLDS`;
- `quiet_beta`, `quiet_error`: the lowest threshold of each instrument with **no** false alarm in the clean race, and `quiet_leads`: a dict `{"beta": ..., "error": ...}` of their lead times;
- `fig`, `ax`: lead time (y) against false-alarm steps (x), one marker per threshold, the two instruments in different colours with a legend, axis labels and a title (leave out thresholds with no lead time);
- `best_trigger`: two or three sentences: which instrument and threshold would you use, and for what: to count, or to act? Why can one crash and one clean race not settle it (the lesson's warning trigger on the 24 training cars)?

**Prediction.** Lower thresholds warn earlier and more often wrongly.
The clean race's peaks (Exercise 5) set the quiet thresholds: the lowest β threshold above the clean peak, and likewise for the line error, and the β warning should lead the line error by most of a second.

In [ ]:
BETA_THRESHOLDS = (4.0, 5.0, 6.0, 8.0, 10.0, 12.0)  # deg
ERROR_THRESHOLDS = (1.0, 1.2, 1.4, 1.6, 1.8)  # m


def false_alarms(log, name, threshold):
    # the steps of a race (above 5 m/s) over the threshold, and the runs of consecutive such steps
    over = (np.abs(log[name]) > threshold) & (log["speed"] > 5.0)
    starts = over & ~np.concatenate(([False], over[:-1]))
    return int(np.sum(over)), int(np.sum(starts))


def threshold_table(name, thresholds):
    return {threshold: (lead_time(crash_log, name, threshold, t_crash), *false_alarms(clean_log, name, threshold))
            for threshold in thresholds}


beta_table = threshold_table("beta", BETA_THRESHOLDS)
error_table = threshold_table("error", ERROR_THRESHOLDS)
quiet_beta = min(threshold for threshold, row in beta_table.items() if row[1] == 0)
quiet_error = min(threshold for threshold, row in error_table.items() if row[1] == 0)
quiet_leads = {"beta": beta_table[quiet_beta][0], "error": error_table[quiet_error][0]}

fig, ax = plt.subplots(figsize=(7, 4))
for table, name, marker in ((beta_table, "sideslip beta", "o"), (error_table, "line error", "s")):
    shown = [threshold for threshold, row in table.items() if row[0] is not None]
    ax.scatter([table[threshold][1] for threshold in shown], [table[threshold][0] for threshold in shown],
               marker=marker, label=name)
ax.set_xlabel("false-alarm steps in the clean race (car 0)")
ax.set_ylabel("lead time before car 16's crash (s)")
ax.set_title("Warning thresholds: how early, and how often wrongly")
ax.legend()
plt.show()
best_trigger = (
    f"The sideslip at {quiet_beta:g} to 8 deg: it was quiet in the clean race and warned {quiet_leads['beta']:.2f} to "
    f"{beta_table[8.0][0]:.2f} s before the crash, where the quiet line-error thresholds warned only "
    f"{quiet_leads['error']:.2f} s ahead (at {quiet_error:g} m), too late to do anything at racing speed, and below "
    "the quiet thresholds both fire in a clean race too; I would use it to count, as the shipped agent does, not to act, since the lesson's fallback on it at 8 deg "
    "saved no car. "
    "One crash and one clean race give one lead time and one false-alarm count: another clean car or seed may cross "
    "6 deg, another crash may give no warning at all (car 20 gave none on the line error), so a threshold has to be "
    "measured on the whole family, as the lesson's 24 training cars did."
)

for name, table in (("beta", beta_table), ("error", error_table)):
    for threshold, row in table.items():
        print(f"{name} > {threshold}: lead {row[0]}, false-alarm steps {row[1]}, episodes {row[2]}")
print("quiet:", quiet_beta, quiet_error, quiet_leads)
print(best_trigger)

# independent checks: false alarms vanish at the clean race's peak, and lead times shrink as the threshold rises
assert beta_table[8.0][0] == lead_beta and error_table[1.6][0] == lead_error
for table, peak in ((beta_table, peak_beta_clean), (error_table, peak_error_clean)):
    for threshold, (lead, steps, episodes) in table.items():
        assert (steps == 0) == (threshold >= peak) and episodes <= steps
    leads = [table[threshold][0] for threshold in sorted(table)]
    assert all(a >= b for a, b in zip(leads, leads[1:]))
assert quiet_beta == 6.0 and quiet_error == 1.4
assert beta_table[5.0][1:] == (1, 1) and abs(quiet_leads["beta"] - 1.25) <= TICK
assert abs(quiet_leads["error"] - 0.30) <= TICK and quiet_leads["error"] < quiet_leads["beta"]  # the line error warns last

## Exercise 7 - A fallback for Unit 10, on the lag-2 car

The lag-2 car, `carla_like(noise=True, lag_ticks=2)` (`race(source, lag_ticks=2)`), has **the lag Unit 8 identified, plus one tick**, and is otherwise the unperturbed car.
Unit 10's MPC crash-loops on it; this exercise gives Unit 10's agent its own fallback, and races it there.

Load Unit 10 at 0.54: `with shim_installed(): Unit10At054, _ = load_submission(variant(UNIT_10, "u10-0.54", MU_FRACTION=0.54, **NO_TIMING))`.
The agent's constants you need: `TARGET_SPEED = 45.0` m/s, `A_DEC = 5.0` and `A_ACC = 4.0` m/s² (Unit 7's profile), `MPC_LAG = 1` (Unit 10), and pure pursuit's lookahead `min(max(PP_K * v, PP_MIN), PP_MAX)` with `PP_K = 0.5` s, `PP_MIN = 3.0` m and `PP_MAX = 40.0` m.

Write:

- `corner_speed(kappa, a_lat)`: √(a_lat / |κ|), but at most `TARGET_SPEED` (and `TARGET_SPEED` when |κ| ≤ 1e-9);
- `plan_profile(cap, segment_length, a_dec, a_acc)`: Unit 7's forward–backward profile: starting from the caps, the backward pass v_i ≤ √(v_{i+1}² + 2 a_dec ds_i) walking the lap backwards, then the forward pass v_{i+1} ≤ √(v_i² + 2 a_acc ds_i), each pass **twice** around the lap (it is a loop);
- `class MyFallback(Unit10At054)` with:
  - `__init__`: set `self.scale = 1.0` **before** calling `super().__init__(*args, **kwargs)` (the parent's constructor plans its profile with `lateral_limit`), then Exercise 4's memory, `self.respawns = 0`, `self.reanchor = False`, `self.mode = "primary"`, and append itself to a list `fallbacks`;
  - `lateral_limit(self, section)`: `self.scale` times the parent's (`super().lateral_limit(section)`);
  - `replan(self)`: `self.cap` from `corner_speed(self.curvature[i], self.lateral_limit(self.section[i]))` for every waypoint i, and `self.profile = plan_profile(self.cap, self.segment_length, 5.0, 4.0)`;
  - `fall_back(self)`: clear the MPC's memory (`self.mpc_queue = [0.0] * 1`, `self.mpc_last = 0.0`, `self.plan = None`), set `self.reanchor = True` and `self.mode = "fallback"`, multiply `self.scale` by 0.86, and `replan()`;
  - `mpc_command(self, yaw, speed)`: in fallback mode, the agent's own pure pursuit from the location sensor's reading, `self.pursuit(x, y, yaw, speed)`; otherwise the parent's `mpc_command`;
  - `async def step(self)`: count the step and apply Exercise 4's edge rule to the collision reading; on a respawn, count it and `fall_back()`; otherwise, if `self.reanchor` is set, re-anchor (`self.current_index = self.nearest_index(location)`, the location sensor's reading) and clear it; then `await super().step()`.

Define:

- `corner_speed`, `plan_profile`, `MyFallback`, `fallbacks`: the functions, the class and the list;
- `profile_matches`: `True` when a fresh `MyFallback(load_track("test-circuit").waypoints, None)` has the same profile (`np.allclose`, `rtol=1e-12`) before and after calling its `replan()` (your profile is Unit 7's);
- `lookaheads`: a dict from speed (5.0, 15.0, 30.0, 45.0 m/s) to the pursuit lookahead (m);
- `plain_lag2`: Unit 10 at 0.54 on the lag-2 car with `max_seconds=100.0`, and `plain_respawns`: its respawn times;
- `my_lag2`: a fresh `MyFallback` on the lag-2 car (`max_seconds=450.0`), and `my_agent`: its agent (the **last** entry of `fallbacks`: `profile_matches` builds one too); `survives`: `True` when it finished with exactly one collision and `my_agent.mode == "fallback"`;
- `known_lag`: Unit 10 at 0.54 with `MPC_LAG=2` on the lag-2 car (`max_seconds=450.0`), and `known_lag_clean`: `clean(known_lag)`;
- `fallback_explained`: two or three sentences: why does the fallback change the controller instead of only slowing down, and why is `MPC_LAG = 2` not a fix you could ship to CARLA?

**Prediction.** `corner_speed` and `plan_profile` are Unit 7's, so the profile is unchanged by `replan()` at scale 1.0.
The lookaheads are 3.0 m (the minimum), 7.5, 15.0 and 22.5 m.
Unit 10 at 0.54 crash-loops on the lag-2 car (about every 31 s); my fallback should crash once at about 30.5 s and finish in pure pursuit, like the lesson's `agents/unit-11.py` (321.65 s), since it is the same fallback written as a subclass; and Unit 10 with `MPC_LAG = 2` should be clean, as the probe record found.

In [ ]:
TARGET_SPEED, A_DEC, A_ACC = 45.0, 5.0, 4.0  # m/s, m/s^2, m/s^2: Unit 7's top speed and profile
PP_K, PP_MIN, PP_MAX = 0.5, 3.0, 40.0  # s, m, m: pure pursuit's lookahead


def corner_speed(kappa, a_lat):
    if abs(kappa) <= 1e-9:
        return TARGET_SPEED
    return min(math.sqrt(a_lat / abs(kappa)), TARGET_SPEED)


def plan_profile(cap, segment_length, a_dec, a_acc):
    n = len(cap)
    v = list(cap)
    for _ in range(2):  # backward pass (braking), twice around the loop
        for i in range(n - 1, -1, -1):
            v[i] = min(v[i], math.sqrt(v[(i + 1) % n] ** 2 + 2 * a_dec * segment_length[i]))
    for _ in range(2):  # forward pass (accelerating), twice around the loop
        for i in range(n):
            v[(i + 1) % n] = min(v[(i + 1) % n], math.sqrt(v[i] ** 2 + 2 * a_acc * segment_length[i]))
    return v


with shim_installed():
    Unit10At054, _ = load_submission(variant(UNIT_10, "u10-0.54", MU_FRACTION=0.54, **NO_TIMING))
fallbacks = []


class MyFallback(Unit10At054):
    def __init__(self, *args, **kwargs):
        self.scale = 1.0  # before the parent's constructor, which plans with lateral_limit
        super().__init__(*args, **kwargs)
        self.steps = 0
        self.previous_reading = None
        self.last_handled = None
        self.respawns = 0
        self.reanchor = False
        self.mode = "primary"
        fallbacks.append(self)

    def lateral_limit(self, section):
        return self.scale * super().lateral_limit(section)

    def replan(self):
        self.cap = [corner_speed(self.curvature[i], self.lateral_limit(self.section[i])) for i in range(len(self.xs))]
        self.profile = plan_profile(self.cap, self.segment_length, 5.0, 4.0)

    def fall_back(self):
        self.mpc_queue = [0.0] * 1
        self.mpc_last = 0.0
        self.plan = None
        self.reanchor = True
        self.mode = "fallback"
        self.scale *= 0.86
        self.replan()

    def mpc_command(self, yaw, speed):
        if self.mode == "fallback":
            location = self.location_sensor.get_last_gym_observation()
            return self.pursuit(float(location[0]), float(location[1]), yaw, speed)
        return super().mpc_command(yaw, speed)

    async def step(self):
        self.steps += 1
        reading = tuple(float(value) for value in self.collision_sensor.get_last_gym_observation())
        edge = float(np.linalg.norm(reading)) > 100.0 and reading != self.previous_reading
        self.previous_reading = reading
        if edge and (self.last_handled is None or self.steps - self.last_handled >= 40):
            self.last_handled = self.steps
            self.respawns += 1
            self.fall_back()
        elif self.reanchor:
            self.current_index = self.nearest_index(self.location_sensor.get_last_gym_observation())
            self.reanchor = False
        await super().step()


probe = MyFallback(load_track("test-circuit").waypoints, None)
before = list(probe.profile)
probe.replan()
profile_matches = bool(np.allclose(before, probe.profile, rtol=1e-12))
lookaheads = {v: min(max(PP_K * v, PP_MIN), PP_MAX) for v in (5.0, 15.0, 30.0, 45.0)}

plain_lag2 = race(variant(UNIT_10, "u10-0.54", MU_FRACTION=0.54, **NO_TIMING), lag_ticks=2, max_seconds=100.0)
plain_respawns = plain_lag2.respawn_times
my_lag2 = race(MyFallback, lag_ticks=2, max_seconds=450.0)
my_agent = fallbacks[-1]
survives = my_lag2.finished and my_lag2.collisions == 1 and my_agent.mode == "fallback"
known_lag = race(variant(UNIT_10, "u10-0.54-lag2", MU_FRACTION=0.54, MPC_LAG=2, **NO_TIMING), lag_ticks=2,
                 max_seconds=450.0)
known_lag_clean = clean(known_lag)
fallback_explained = (
    "Slowing down helps only if the speed plan was the problem; on the lag-2 car it is the MPC's model of the car "
    f"that is wrong (it plans for one tick of lag and gets two), so it crashes at the same corner every "
    f"{np.mean(np.diff(plain_respawns)):.1f} s at any fraction, and only a controller that does not depend on that "
    f"model, pure pursuit, gets the car home ({my_lag2.total_time:.2f} s with one crash). "
    f"MPC_LAG = 2 is clean here ({known_lag.total_time:.2f} s) only because I know this car's lag exactly; on CARLA "
    "the lag is known only as well as Unit 8 measured it (one tick), and the lesson's probe record found MPC_LAG = 2 "
    "crashing on a lag-2 car whose throttle or steering was a few per cent off too, so it trades one model error for "
    "another instead of surviving the error."
)

print(f"profile matches {profile_matches}; lookaheads {lookaheads}")
print(f"plain lag 2: respawns {plain_respawns}, finished {plain_lag2.finished}")
print(f"my fallback: {my_lag2.total_time:.2f} s, collisions {my_lag2.collisions} at {my_lag2.respawn_times}, "
      f"finished {my_lag2.finished}, mode {my_agent.mode}, respawns {my_agent.respawns}, scale {my_agent.scale}, "
      f"clean-rule line error {line_error(my_lag2):.2f} m")
print(f"known lag: {known_lag.total_time:.2f} s, collisions {known_lag.collisions}, clean {known_lag_clean}, "
      f"line error {line_error(known_lag):.2f} m")
print(fallback_explained)

# independent checks: my profile against the shipped agent's own (Unit 7's), the lookahead by hand, and the
# lesson's agents/unit-11.py on the same car (321.65 s, one collision at 30.55 s): the same fallback, written by hand
assert profile_matches and lookaheads == {5.0: 3.0, 15.0: 7.5, 30.0: 15.0, 45.0: 22.5}
assert corner_speed(0.0, 8.0) == 45.0 and abs(corner_speed(0.02, 8.0) - 20.0) <= 1e-12
assert not plain_lag2.finished and len(plain_respawns) >= 3 and abs(plain_respawns[0] - 30.55) <= TICK
assert survives and my_agent.respawns == 1 and abs(my_agent.scale - 0.86) <= 1e-12
assert my_lag2.respawn_times == plain_respawns[:1]  # the same MPC up to its first crash
assert abs(my_lag2.total_time - 321.65) <= TICK
assert known_lag_clean and abs(known_lag.total_time - 280.00) <= TICK

## Exercise 8 - A robust fraction, by the margin rule

Re-tune the fraction the way Units 7 to 10 did, on a small training family, and check it on a held-out car.

- **The training family:** the three lowest-grip training cars, `grip_order[:3]` (Exercise 3).
- **The edge:** for f = 0.58, then 0.56, then 0.54, race Unit 10 at f (`variant(UNIT_10, ..., MU_FRACTION=f, **NO_TIMING)`) on the family's cars in grip order, with race seed **0** for every car (`seed=0`, as the probe record did) and `max_seconds=300.0`.
  Stop racing a fraction at its first car that is not clean (the fraction is then not clean on the family), and stop the search at the first fraction that is clean on all three: that is the **edge**.
- **The margin rule:** ship one grid step (0.02) inside the edge.
- **The noise:** the book's tests race car k with race seed k mod 5, not 0.
  Race the edge fraction once more on car 8 (a family car that is clean at the edge with seed 0) with its own race seed, 3 (`race(source, 8, max_seconds=300.0)`).
- **The held-out check:** race the shipped fraction on the lowest-grip **held-out** car (from `factors`), race seed 0, `max_seconds=450.0`.

Define:

- `small_family`: the three cars;
- `edge_races`: a dict from (f, k) to the result, for every race of the search, and `family_clean`: a dict from each fraction searched to `True` or `False`;
- `edge`: the edge fraction, and `shipped`: the margin rule's fraction (`round(edge - 0.02, 2)`);
- `noise_race`: the race of car 8 at the edge with race seed 3, and `edge_moves`: `True` when it is not clean;
- `held_out_car`, `held_out_race`, `held_out_clean`: the held-out car, its race, and `clean(held_out_race)`;
- `races_run`: how many races this exercise raced;
- `is_it_enough`: two or three sentences: a 3-car family picked by grip, one seed per car: what could this procedure miss, and what did the lesson's 24 + 24 cars and the seed2026 finding add?

**Prediction.** The lesson's probe record (race seed 0 on all training cars) found the edge at 0.56, so on these three cars I expect 0.58 to fail on car 16 and 0.56 to be clean on all three: edge 0.56, ship 0.54.
With its own race seed 3, car 8 at 0.56 is one of the book's tests' crashes (474.00 s), so the edge moves.
The held-out check at 0.54 should be clean: the lesson's 24 held-out races at 0.54 were.

In [ ]:
small_family = grip_order[:3]
edge_races, family_clean = {}, {}
edge = None
for f in (0.58, 0.56, 0.54):
    source = variant(UNIT_10, f"u10-{f}", MU_FRACTION=f, **NO_TIMING)
    family_clean[f] = True
    for k in small_family:
        edge_races[(f, k)] = race(source, k, seed=0, max_seconds=300.0)
        if not clean(edge_races[(f, k)]):
            family_clean[f] = False
            break
    if family_clean[f]:
        edge = f
        break
shipped = round(edge - 0.02, 2)

noise_race = race(variant(UNIT_10, f"u10-{edge}", MU_FRACTION=edge, **NO_TIMING), 8, max_seconds=300.0)
edge_moves = not clean(noise_race)
held_out_car = min(HELD_OUT, key=lambda k: factors[k]["grip"])
held_out_race = race(variant(UNIT_10, f"u10-{shipped}", MU_FRACTION=shipped, **NO_TIMING), held_out_car, seed=0,
                     max_seconds=450.0)
held_out_clean = clean(held_out_race)
races_run = len(edge_races) + 2
is_it_enough = (
    "Three cars chosen by grip and one seed each can miss a car whose danger is not its grip (a strong throttle, a "
    "weak brake, a different steering gain) and a crash that only some draws of the noise produce: here car 8 is "
    f"clean at the edge {edge} on seed 0 and crashes on its own seed 3, so the edge itself moves with the noise. "
    "The lesson's 24 training cars on their own seeds k mod 5 measured the family's crash rate and expected race time "
    "at each fraction, the 24 held-out cars checked 0.54 on cars never tuned on (24 of 24 clean), and the seed2026 "
    "finding showed that even that margin does not carry over to another track."
)

for (f, k), result in edge_races.items():
    print(f"f = {f}, car {k} (seed 0): clean {clean(result)}, {result.total_time:.2f} s, collisions "
          f"{result.collisions} at {result.respawn_times}, line error {line_error(result):.2f} m")
print(f"edge {edge}, shipped {shipped}")
print(f"noise race (car 8, seed 3, f = {edge}): clean {clean(noise_race)}, {noise_race.total_time:.2f} s, "
      f"collisions at {noise_race.respawn_times}")
print(f"held-out car {held_out_car} (grip {factors[held_out_car]['grip']:.3f}): clean {held_out_clean}, "
      f"{held_out_race.total_time:.2f} s, line error {line_error(held_out_race):.2f} m; races run {races_run}")
print(is_it_enough)

# independent checks: the search's own record, and the lesson's probe record (edge 0.56 on seed 0, ship 0.54)
assert small_family == [16, 18, 8]
assert family_clean == {0.58: False, 0.56: True} and edge == 0.56 and shipped == 0.54
assert list(edge_races) == [(0.58, 16), (0.56, 16), (0.56, 18), (0.56, 8)]
assert edge_races[(0.58, 16)].collisions == 1 and abs(edge_races[(0.58, 16)].respawn_times[0] - 99.90) <= TICK
assert edge_moves and abs(noise_race.respawn_times[0] - 195.55) <= TICK
assert held_out_car == 35 and factors[35]["grip"] == min(factors[k]["grip"] for k in HELD_OUT)
assert held_out_clean and abs(held_out_race.total_time - 280.25) <= TICK and races_run == 6

## Exercise 9 - Challenge: brake on the warning

The lesson acted on the sideslip warning by **falling back** (pure pursuit, a lower speed plan), and it did not save a single car.
Try a different reaction: **brake**.
While the agent is in its primary mode, above 5 m/s, and |β| > `BRAKE_BETA`, lower the speed goal to `speed - BRAKE_DROP`; otherwise change nothing.

The speed goal comes from the agent's `profile_goal(self, speed)` method, called after the warning is computed in each step (so `self.beta` is this step's).
Make a function `braking(source, brake_beta, brake_drop)` that returns a subclass of the submission's class whose `profile_goal` returns `min(super().profile_goal(speed), speed - brake_drop)` when the condition holds, and counts those steps in `self.brakings`.

Use `agents/unit-11.py` at 0.58 (`variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)`) on car 20, whose crash at 0.58 comes at about 99 s on race seed 0.
Is a save on one seed a save?
Race car 20 on seeds 0, 1 and 2 with `repeat_races(..., "test-circuit", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True, perturb=20), max_seconds=110.0)`, once with the plain file and once with `braking(source, 8.0, 3.0)`.

Define:

- `braking`: the function;
- `plain_runs`, `braked_runs`: the two `RepeatResult` objects;
- `plain_crashed`, `braked_crashed`: dicts from seed to `True` when that race had a collision;
- `saved_seeds`: the seeds on which the plain file crashed and the braking one did not;
- `crash_times`: a dict from seed to `(plain crash time, braked crash time)` where both crashed;
- `verdict`: two or three sentences: did braking save car 20 on any seed? If it had saved one seed, why would that still not be evidence (Unit 1's rule, the lesson's 24 cars), and what would you race next?

**Prediction.** The lesson's fallback on the β warning came 0.75 s before car 20's crash and did not save it; braking 3 m/s below the current speed in that time removes only a little speed, so I expect braking not to save car 20 on seed 0, perhaps to delay the crash a little; seeds 1 and 2 may not crash at all before 110 s.

In [ ]:
def braking(source, brake_beta, brake_drop):
    with shim_installed():
        Agent, _ = load_submission(source)

    class Braking(Agent):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.brakings = 0

        def profile_goal(self, speed):
            goal = super().profile_goal(speed)
            if self.mode == "primary" and speed > 5.0 and abs(self.beta) > brake_beta:
                self.brakings += 1
                return min(goal, speed - brake_drop)
            return goal

    return Braking


source = variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)
SEEDS = (0, 1, 2)
plain_runs = repeat_races(source, "test-circuit", n=3, seeds=SEEDS, car_factory=lambda: carla_like(noise=True, perturb=20),
                          max_seconds=110.0)
braked_runs = repeat_races(braking(source, 8.0, 3.0), "test-circuit", n=3, seeds=SEEDS,
                           car_factory=lambda: carla_like(noise=True, perturb=20), max_seconds=110.0)
plain_crashed = {seed: result.collisions > 0 for seed, result in zip(SEEDS, plain_runs.results)}
braked_crashed = {seed: result.collisions > 0 for seed, result in zip(SEEDS, braked_runs.results)}
saved_seeds = [seed for seed in SEEDS if plain_crashed[seed] and not braked_crashed[seed]]
crash_times = {seed: (plain.respawn_times[0], braked.respawn_times[0])
               for seed, plain, braked in zip(SEEDS, plain_runs.results, braked_runs.results)
               if plain_crashed[seed] and braked_crashed[seed]}
verdict = (
    f"No: braking saved car 20 on no seed; it crashed on seeds 0 and 2 either way, {crash_times[0][0] - crash_times[0][1]:.2f} "
    f"and {crash_times[2][0] - crash_times[2][1]:.2f} s earlier with braking, and seed 1 did not crash before 110 s "
    "with or without it, so it shows nothing. "
    "Had it saved one seed, that would be one race, and Unit 1's rule says one race partly ranks the noise: the crash "
    "comes and goes with the seed (seed 1 here), so a save must hold on several seeds and across the lesson's 24 "
    "training cars without slowing the clean ones. "
    "Next I would race a larger drop or an earlier threshold (6 deg, Exercise 6's quiet one) on cars 8, 16 and 20 over "
    "five seeds each, and then every training car at 0.58 for the expected race time."
)

for seed, plain, braked in zip(SEEDS, plain_runs.results, braked_runs.results):
    print(f"seed {seed}: plain crashes at {plain.respawn_times}, braked at {braked.respawn_times}")
print("saved:", saved_seeds, "crash times:", crash_times)
print(verdict)

# independent checks: the plain file is the lesson's car-20 race on seed 0 (the crash at 99.10 s)
assert abs(plain_runs.results[0].respawn_times[0] - 99.10) <= TICK
assert plain_crashed == {0: True, 1: False, 2: True} and braked_crashed == {0: True, 1: False, 2: True}
assert saved_seeds == [] and sorted(crash_times) == [0, 2]
assert all(braked < plain for plain, braked in crash_times.values())  # braking did not even delay the crash
assert not plain_runs.all_finished and not braked_runs.all_finished  # 110 s is not a race

## Exercise 10 - Challenge: a fraction for `test-circuit-seed2026`

The lesson's finding: the 0.54 that is clean on all 48 perturbed cars on `test-circuit` crashes on 4 of 12 on `test-circuit-seed2026` (premise U11.8: cars 6, 8, 9 and 10, race seed 0).
Find a fraction that is clean there.

Solve `test-circuit-seed2026`'s line (`seed2026 = load_track("test-circuit-seed2026")`, `solve_line(seed2026)`) and write it, with `MEASURED_MU_G_2026`, into `agents/unit-11.py` (`variant(UNIT_11, ..., LINE_CONTROL=tuple(float(v) for v in control), MU_G=MEASURED_MU_G_2026, MU_FRACTION=f, **NO_TIMING)`).
For f = 0.52, then 0.50, then 0.48, race cars 6 and 9 (`race(source, k, track="test-circuit-seed2026", seed=0, max_seconds=300.0)`), stopping at the first fraction clean on both.

Define:

- `seed2026_control`, `seed2026_offsets`: the line's control values and its offsets (`spline_basis(len(seed2026)) @ seed2026_control`);
- `seed2026_races`: a dict from (f, k) to the result, `seed2026_clean`: a dict from (f, k) to `clean(result, seed2026_offsets)`;
- `seed2026_fraction`: the first fraction clean on both cars;
- `slower_by`: the mean race time of your two clean races at that fraction minus 282.55 s (the premise's median over cars 0-11 at 0.54 there);
- `one_fraction`: two or three sentences: Checkpoint B forbids tuning per track. Would you ship your fraction everywhere, given what one step costs on `test-circuit` (Exercise 2), and what would you race before deciding?

**Prediction.** At 0.54 cars 6 and 9 crash on `test-circuit-seed2026`; a step or two lower should clear them, at a cost of about 2.5 s per step, so I expect 0.52 or 0.50, a few seconds slower than 282.55 s.

In [ ]:
seed2026 = load_track("test-circuit-seed2026")
seed2026_control = solve_line(seed2026)
seed2026_offsets = spline_basis(len(seed2026)) @ seed2026_control
seed2026_races, seed2026_clean = {}, {}
seed2026_fraction = None
for f in (0.52, 0.50, 0.48):
    source = variant(UNIT_11, f"u11-seed2026-{f}", LINE_CONTROL=tuple(float(v) for v in seed2026_control),
                     MU_G=MEASURED_MU_G_2026, MU_FRACTION=f, **NO_TIMING)
    for k in (6, 9):
        seed2026_races[(f, k)] = race(source, k, track="test-circuit-seed2026", seed=0, max_seconds=300.0)
        seed2026_clean[(f, k)] = clean(seed2026_races[(f, k)], seed2026_offsets)
    if seed2026_clean[(f, 6)] and seed2026_clean[(f, 9)]:
        seed2026_fraction = f
        break
slower_by = None
if seed2026_fraction is not None:
    slower_by = np.mean([seed2026_races[(seed2026_fraction, k)].total_time for k in (6, 9)]) - 282.55
one_fraction = (
    f"Not from this alone: {seed2026_fraction} is clean on cars 6 and 9 here, {slower_by:.2f} s slower than the "
    "premise's 0.54 median, but shipping it everywhere costs about one step's 2.5 s on every clean test-circuit race "
    "(Exercise 2), which pays only if it prevents more than about one crash in 78 races. "
    "Before deciding I would race it on all 12 cars of the premise and their other seeds on test-circuit-seed2026, "
    "and on the training tracks' perturbed cars, comparing the expected race time of 0.52 and 0.54 over all of them."
)

for (f, k), result in seed2026_races.items():
    print(f"f = {f}, car {k}: clean {seed2026_clean[(f, k)]}, {result.total_time:.2f} s, collisions at "
          f"{result.respawn_times}, line error {line_error(result, seed2026_offsets):.2f} m")
print("fraction", seed2026_fraction, "slower by", slower_by)
print(one_fraction)

# independent checks: the line stays inside the track, and both races are clean against their own line
assert len(seed2026_offsets) == len(seed2026) and np.max(np.abs(seed2026_offsets)) <= np.max(seed2026.half_width) - 2.5 + 1e-9
assert seed2026_fraction == 0.52 and list(seed2026_races) == [(0.52, 6), (0.52, 9)]
assert all(seed2026_clean.values())
assert abs(seed2026_races[(0.52, 6)].total_time - 284.80) <= TICK and abs(seed2026_races[(0.52, 9)].total_time - 285.30) <= TICK
assert abs(slower_by - 2.50) <= TICK